In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_ROOT = Path("../../data")
RAW_RATINGS_FILE = DATA_ROOT / "raw" / "movie_data_1" / "ratings.csv"
LINKS_FILE = DATA_ROOT / "raw" / "movie_data_1" / "links.csv"
ARTIFACT_DIR = Path("../../ml/artifacts/content_recommendation_v2")
METADATA_FILE = ARTIFACT_DIR / "movie_metadata.csv"
FEATURE_ARRAYS_FILE = ARTIFACT_DIR / "feature_arrays.npz"
OUTPUT_FILE = DATA_ROOT / "processed" / "interactions_clean.parquet"

CHUNK_SIZE = 5_000_000
MIN_USERS = 20_000
MAX_USERS = 50_000
RANDOM_STATE = 42

# MovieLens rating scale: 0.5–5.0
# Positive threshold: >= 4.0 (equivalent to "liked")
POSITIVE_THRESHOLD = 4.0

In [2]:
# Load movie catalog (TMDB IDs) and MovieLens → TMDB ID mapping
catalog = pd.read_csv(METADATA_FILE, usecols=["movie_index", "id"])
catalog["id"] = pd.to_numeric(catalog["id"], errors="coerce").dropna().astype("int64")
catalog_ids = set(catalog["id"].tolist())

links = pd.read_csv(LINKS_FILE)
links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce")
links = links.dropna(subset=["tmdbId"])
links["movieId"] = links["movieId"].astype("int64")
links["tmdbId"] = links["tmdbId"].astype("int64")
ml_to_tmdb = dict(zip(links["movieId"], links["tmdbId"]))

print(f"Catalog movies (TMDB): {len(catalog_ids):,}")
print(f"MovieLens → TMDB links: {len(ml_to_tmdb):,}")

# Chunked reading of raw ratings, mapping movieId → tmdbId and filtering to catalog
chunk_reader = pd.read_csv(
    RAW_RATINGS_FILE,
    usecols=["userId", "movieId", "rating"],
    dtype={"userId": "int64", "movieId": "int64", "rating": "float32"},
    chunksize=CHUNK_SIZE,
)

Catalog movies (TMDB): 30,543
MovieLens → TMDB links: 45,624


In [3]:
# First pass: count users per movie after mapping and filtering
rows_total = 0
rows_after_mapping = 0
rows_after_catalog = 0
user_counts = pd.Series(dtype="int64")

for chunk in chunk_reader:
    rows_total += len(chunk)

    # Map MovieLens movieId → TMDB tmdbId
    chunk["movie_id"] = chunk["movieId"].map(ml_to_tmdb)
    chunk = chunk.dropna(subset=["movie_id"])
    chunk["movie_id"] = chunk["movie_id"].astype("int64")
    rows_after_mapping += len(chunk)

    # Filter to catalog
    chunk = chunk[chunk["movie_id"].isin(catalog_ids)]
    rows_after_catalog += len(chunk)

    if chunk.empty:
        continue

    chunk_counts = chunk["userId"].value_counts()
    user_counts = user_counts.add(chunk_counts, fill_value=0)

print(f"Total rows scanned: {rows_total:,}")
print(f"Rows after movieId → tmdbId mapping: {rows_after_mapping:,} ({(1 - rows_after_mapping/rows_total)*100:.1f}% dropped)")
print(f"Rows after catalog filter: {rows_after_catalog:,} ({(1 - rows_after_catalog/rows_after_mapping)*100:.1f}% dropped)")
print(f"Unique users with valid ratings: {len(user_counts):,}")

Total rows scanned: 26,024,289
Rows after movieId → tmdbId mapping: 26,010,786 (0.1% dropped)
Rows after catalog filter: 25,555,768 (1.7% dropped)
Unique users with valid ratings: 270,813


In [4]:
# Stratified user sampling across activity tiers
user_counts = user_counts.astype("int64").sort_values(ascending=False)
available_users = len(user_counts)

if available_users == 0:
    raise ValueError("No eligible users found after filtering.")

if available_users <= MIN_USERS:
    target_users = available_users
else:
    target_users = min(MAX_USERS, available_users)

if available_users > target_users:
    n_bins = min(10, int(user_counts.nunique()))
    if n_bins > 1:
        activity_bins = pd.qcut(user_counts, q=n_bins, labels=False, duplicates="drop")
        rng = np.random.default_rng(RANDOM_STATE)
        selected_user_ids = []

        unique_bins = sorted(activity_bins.dropna().unique())
        remaining_slots = target_users

        for i, bin_id in enumerate(unique_bins):
            bin_members = user_counts.index[activity_bins == bin_id]
            bins_left = len(unique_bins) - i
            take = min(len(bin_members), max(1, remaining_slots // bins_left))
            chosen = rng.choice(bin_members.to_numpy(), size=take, replace=False)
            selected_user_ids.extend(chosen.tolist())
            remaining_slots -= take

        if len(selected_user_ids) < target_users:
            selected_index = pd.Index(selected_user_ids)
            additional = user_counts.index.difference(selected_index)[: target_users - len(selected_user_ids)]
            selected_user_ids.extend(additional.tolist())
    else:
        selected_user_ids = user_counts.index[:target_users].tolist()
else:
    selected_user_ids = user_counts.index.tolist()

selected_user_ids = set(selected_user_ids)
print(f"Available users: {available_users:,}")
print(f"Selected users: {len(selected_user_ids):,}")

Available users: 270,813
Selected users: 50,000


In [5]:
# Second pass: collect clean interactions for selected users
clean_chunks = []
chunk_reader = pd.read_csv(
    RAW_RATINGS_FILE,
    usecols=["userId", "movieId", "rating"],
    dtype={"userId": "int64", "movieId": "int64", "rating": "float32"},
    chunksize=CHUNK_SIZE,
)

for chunk in chunk_reader:
    # Map MovieLens movieId → TMDB tmdbId
    chunk["movie_id"] = chunk["movieId"].map(ml_to_tmdb)
    chunk = chunk.dropna(subset=["movie_id"])
    chunk["movie_id"] = chunk["movie_id"].astype("int64")

    # Filter to catalog and selected users
    chunk = chunk[chunk["movie_id"].isin(catalog_ids)]
    chunk = chunk[chunk["userId"].isin(selected_user_ids)]

    if chunk.empty:
        continue

    chunk = chunk[["userId", "movie_id", "rating"]].copy()
    chunk = chunk.rename(columns={"userId": "user_id"})
    chunk["rating"] = chunk["rating"].astype("float32")
    clean_chunks.append(chunk)

if not clean_chunks:
    raise ValueError("No rows remained after applying all filters.")

interactions_clean = pd.concat(clean_chunks, ignore_index=True)
interactions_clean = interactions_clean.astype(
    {"user_id": "int64", "movie_id": "int64", "rating": "float32"}
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
interactions_clean.to_parquet(OUTPUT_FILE, index=False)

print(f"Output rows: {len(interactions_clean):,}")
print(f"Unique users: {interactions_clean['user_id'].nunique():,}")
print(f"Unique movies: {interactions_clean['movie_id'].nunique():,}")
print(f"Rating distribution:\n{interactions_clean['rating'].value_counts().sort_index()}")
print(f"Saved: {OUTPUT_FILE}")

Output rows: 4,762,544
Unique users: 50,000
Unique movies: 24,376
Rating distribution:
rating
0.5      70026
1.0     153994
1.5      73958
2.0     320070
2.5     232470
3.0     963849
3.5     572473
4.0    1280442
4.5     398012
5.0     697250
Name: count, dtype: int64
Saved: ../../data/processed/interactions_clean.parquet


## Item Features — 1) Catalog and Interaction Aggregates
Build item-level aggregate statistics from the full interaction dataset (before user subsampling) using chunked reads.
These are population-level statistics for the global (non-personalized) reranker.

In [6]:
import ast

ITEM_FEATURES_DIR = DATA_ROOT / "processed"

def _safe_numeric(series, dtype="float64"):
    return pd.to_numeric(series, errors="coerce").astype(dtype)

def _parse_list_field(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed if str(item).strip()]
        except (ValueError, SyntaxError):
            pass
    return [part.strip() for part in text.split(",") if part.strip()]

def _parse_genres(value):
    return _parse_list_field(value)

In [7]:
# Load full movie catalog from artifacts
movie_catalog = pd.read_csv(METADATA_FILE)
movie_catalog["id"] = _safe_numeric(movie_catalog["id"], dtype="Int64")
movie_catalog = movie_catalog.dropna(subset=["id"]).copy()
movie_catalog["id"] = movie_catalog["id"].astype("int64")
catalog_movie_ids = set(movie_catalog["id"].tolist())

# Aggregate user rating stats from FULL raw dataset (all users, not subsampled)
# using chunked reads for memory safety
agg = {}
chunk_reader = pd.read_csv(
    RAW_RATINGS_FILE,
    usecols=["movieId", "rating"],
    dtype={"movieId": "int64", "rating": "float32"},
    chunksize=CHUNK_SIZE,
)

rows_seen = 0
rows_in_catalog = 0

for chunk in chunk_reader:
    rows_seen += len(chunk)

    # Map MovieLens movieId → TMDB tmdbId
    chunk["movie_id"] = chunk["movieId"].map(ml_to_tmdb)
    chunk = chunk.dropna(subset=["movie_id"])
    chunk["movie_id"] = chunk["movie_id"].astype("int64")

    # Filter to catalog
    chunk = chunk[chunk["movie_id"].isin(catalog_movie_ids)]
    rows_in_catalog += len(chunk)

    if chunk.empty:
        continue

    grp = chunk.groupby("movie_id")["rating"].agg(
        rating_count="count",
        rating_sum="sum",
        rating_sq_sum=lambda x: (x.astype("float64") ** 2).sum(),
        positive_count=lambda x: (x >= POSITIVE_THRESHOLD).sum(),
    )

    for movie_id, row in grp.iterrows():
        stats = agg.setdefault(
            int(movie_id),
            {"rating_count": 0, "rating_sum": 0.0, "rating_sq_sum": 0.0, "positive_count": 0},
        )
        stats["rating_count"] += int(row["rating_count"])
        stats["rating_sum"] += float(row["rating_sum"])
        stats["rating_sq_sum"] += float(row["rating_sq_sum"])
        stats["positive_count"] += int(row["positive_count"])

agg_df = pd.DataFrame.from_dict(agg, orient="index")
agg_df.index.name = "id"
agg_df = agg_df.reset_index()

agg_df["avg_user_rating"] = agg_df["rating_sum"] / agg_df["rating_count"]
mean_sq = agg_df["rating_sq_sum"] / agg_df["rating_count"]
variance = (mean_sq - (agg_df["avg_user_rating"] ** 2)).clip(lower=0)
agg_df["rating_std"] = np.sqrt(variance)
agg_df["positive_ratio"] = agg_df["positive_count"] / agg_df["rating_count"]

print(f"Rows scanned: {rows_seen:,}")
print(f"Rows in catalog: {rows_in_catalog:,}")
print(f"Movies with user ratings: {len(agg_df):,}")

Rows scanned: 26,024,289
Rows in catalog: 25,555,768
Movies with user ratings: 30,023


## Item Features — 2) Join Catalog Features and Encode Categorical Fields
Merge rating aggregates with catalog metadata, then create language encoding and genre vectors.

In [8]:
# Merge rating aggregates with full movie catalog
item_features = movie_catalog.merge(agg_df, on="id", how="left")
item_features["rating_count"] = item_features["rating_count"].fillna(0).astype("int64")
item_features["avg_user_rating"] = item_features["avg_user_rating"].fillna(0.0).astype("float32")
item_features["rating_std"] = item_features["rating_std"].fillna(0.0).astype("float32")
item_features["positive_ratio"] = item_features["positive_ratio"].fillna(0.0).astype("float32")

# Catalog-level quality signals
item_features["catalog_score"] = _safe_numeric(item_features["vote_average"]).fillna(0.0).astype("float32")
item_features["catalog_voted_by"] = _safe_numeric(item_features["vote_count"]).fillna(0).astype("int64")
# Popularity rank: higher vote_count → more popular. Rank ascending so rank 1 = most votes.
item_features["popularity_rank"] = item_features["catalog_voted_by"].rank(ascending=False, method="min").astype("int64")

# Language encoding (replaces anime Type/Source)
item_features["original_language"] = item_features["original_language"].fillna("unknown").astype(str).str.strip().str.lower()
item_features["language_encoded"] = pd.Categorical(item_features["original_language"]).codes.astype("int16")

# Genre multi-hot vectors
genre_lists = item_features["genres"].map(_parse_genres)
item_features["n_genres"] = genre_lists.map(len).astype("int16")

unique_genres = sorted({g for gs in genre_lists for g in gs})
for genre in unique_genres:
    col = f"genre_{genre.lower().replace(' ', '_').replace('-', '_').replace('/', '_')}"
    item_features[col] = genre_lists.map(lambda vs, g=genre: int(g in vs)).astype("int8")

print(f"Item features shape after catalog join: {item_features.shape}")
print(f"Movies with user rating data: {(item_features['rating_count'] > 0).sum():,} / {len(item_features):,}")
print(f"Unique genres: {len(unique_genres)}")

Item features shape after catalog join: (31167, 48)
Movies with user rating data: 30,636 / 31,167
Unique genres: 20


## Item Features — 3) Merge Bayesian Scores and Save Artifact
Attach `bayesian_score_norm` from feature_arrays.npz using `movie_index`, select final columns, and write `item_features.parquet`.

In [9]:
# Drop stale column if it exists from a prior run
if "bayesian_score_norm" in item_features.columns:
    item_features = item_features.drop(columns=["bayesian_score_norm"])

if METADATA_FILE.exists() and FEATURE_ARRAYS_FILE.exists():
    # movie_metadata.csv has movie_index (row position for embeddings/feature arrays) and id (TMDB ID)
    artifact_meta = pd.read_csv(METADATA_FILE, usecols=["id", "movie_index"])
    artifact_meta["id"] = _safe_numeric(artifact_meta["id"], dtype="Int64")
    artifact_meta["movie_index"] = _safe_numeric(artifact_meta["movie_index"], dtype="Int64")
    artifact_meta = artifact_meta.dropna(subset=["id", "movie_index"]).copy()
    artifact_meta["id"] = artifact_meta["id"].astype("int64")
    artifact_meta["movie_index"] = artifact_meta["movie_index"].astype("int64")

    feature_arrays = np.load(FEATURE_ARRAYS_FILE)
    if "bayesian_scores_norm" in feature_arrays.files:
        bayes = feature_arrays["bayesian_scores_norm"]
    elif "bayesian_scores" in feature_arrays.files:
        raw = feature_arrays["bayesian_scores"].astype("float64")
        min_v, max_v = raw.min(), raw.max()
        bayes = (raw - min_v) / (max_v - min_v) if max_v > min_v else np.zeros_like(raw)
    else:
        bayes = None

    if bayes is not None and not artifact_meta.empty:
        valid = artifact_meta[artifact_meta["movie_index"].between(0, len(bayes) - 1)].copy()
        valid["bayesian_score_norm"] = bayes[valid["movie_index"].to_numpy()].astype("float32")
        item_features = item_features.merge(
            valid[["id", "bayesian_score_norm"]],
            on="id",
            how="left",
        )
    else:
        item_features["bayesian_score_norm"] = np.nan
else:
    item_features["bayesian_score_norm"] = np.nan

if "bayesian_score_norm" not in item_features.columns:
    item_features["bayesian_score_norm"] = np.nan

item_features["bayesian_score_norm"] = item_features["bayesian_score_norm"].fillna(0.0).astype("float32")

# Rename id → movie_id for downstream consistency
item_features = item_features.rename(columns={"id": "movie_id"})

# Select final columns
keep_cols = [
    "movie_id",
    "avg_user_rating",
    "rating_count",
    "rating_std",
    "positive_ratio",
    "catalog_score",
    "catalog_voted_by",
    "bayesian_score_norm",
    "popularity_rank",
    "language_encoded",
    "n_genres",
] + [c for c in item_features.columns if c.startswith("genre_")]

item_features = item_features[keep_cols].sort_values("movie_id").reset_index(drop=True)

ITEM_FEATURES_DIR.mkdir(parents=True, exist_ok=True)
item_features.to_csv(ITEM_FEATURES_DIR / "item_features.csv", index=False)
item_features.to_parquet(ITEM_FEATURES_DIR / "item_features.parquet", index=False)

print(f"Item feature shape: {item_features.shape}")
print(f"Saved to: {ITEM_FEATURES_DIR}")
print(item_features[["movie_id", "avg_user_rating", "rating_count", "positive_ratio", "bayesian_score_norm"]].head(10))

Item feature shape: (33171, 31)
Saved to: ../../data/processed
   movie_id  avg_user_rating  rating_count  positive_ratio  \
0         2         3.673664           262        0.553435   
1         3         3.770115            87        0.597701   
2         5         3.409031          6090        0.439245   
3         6         2.924862          1271        0.220299   
4        11         4.132299         77045        0.762944   
5        12         3.855151         33887        0.629238   
6        13         4.052926         91921        0.724916   
7        14         4.130704         57879        0.758894   
8        15         4.094047         20830        0.731925   
9        16         3.841610          4571        0.651499   

   bayesian_score_norm  
0             0.582210  
1             0.570331  
2             0.593172  
3             0.545700  
4             0.872583  
5             0.788298  
6             0.889980  
7             0.836045  
8             0.842232  
9   

## User Features (Training Only)
Build user profile aggregates from cleaned interactions and save offline-only features to `data/processed/user_features.parquet`.
These features are used during **training only** — the global reranker does NOT use user features at inference time.

In [10]:
import json

USER_FEATURES_FILE = DATA_ROOT / "processed" / "user_features.parquet"

# Load interactions (use in-memory if available, else read from disk)
if "interactions_clean" in globals() and isinstance(interactions_clean, pd.DataFrame):
    interactions_for_users = interactions_clean[["user_id", "movie_id", "rating"]].copy()
else:
    interactions_for_users = pd.read_parquet(
        OUTPUT_FILE,
        columns=["user_id", "movie_id", "rating"],
    )

interactions_for_users["user_id"] = pd.to_numeric(interactions_for_users["user_id"], errors="coerce")
interactions_for_users["movie_id"] = pd.to_numeric(interactions_for_users["movie_id"], errors="coerce")
interactions_for_users["rating"] = pd.to_numeric(interactions_for_users["rating"], errors="coerce")
interactions_for_users = interactions_for_users.dropna(subset=["user_id", "movie_id", "rating"]).copy()
interactions_for_users["user_id"] = interactions_for_users["user_id"].astype("int64")
interactions_for_users["movie_id"] = interactions_for_users["movie_id"].astype("int64")
interactions_for_users["rating"] = interactions_for_users["rating"].astype("float32")

# --- Basic user stats ---
user_stats = interactions_for_users.groupby("user_id")["rating"].agg(
    user_avg_rating="mean",
    user_rating_count="count",
    user_rating_std="std",
).reset_index()
user_stats["user_rating_std"] = user_stats["user_rating_std"].fillna(0.0)

# --- Merge with catalog metadata for genre/language/popularity ---
item_meta = movie_catalog[["id", "genres", "original_language", "vote_count"]].copy()
item_meta = item_meta.rename(columns={"id": "movie_id"})
item_meta["movie_id"] = pd.to_numeric(item_meta["movie_id"], errors="coerce")
item_meta = item_meta.dropna(subset=["movie_id"]).copy()
item_meta["movie_id"] = item_meta["movie_id"].astype("int64")
item_meta["vote_count"] = pd.to_numeric(item_meta["vote_count"], errors="coerce").fillna(0)

merged = interactions_for_users.merge(item_meta, on="movie_id", how="left")

# --- User average item popularity (by vote_count) ---
user_popularity = (
    merged.groupby("user_id", as_index=False)["vote_count"]
    .mean()
    .rename(columns={"vote_count": "user_avg_item_popularity"})
)
user_popularity["user_avg_item_popularity"] = user_popularity["user_avg_item_popularity"].fillna(0.0)

# --- Genre distribution per user ---
merged["genre_list"] = merged["genres"].map(_parse_genres)
genre_exploded = merged[["user_id", "genre_list"]].explode("genre_list")
genre_exploded = genre_exploded.dropna(subset=["genre_list"])

if genre_exploded.empty:
    user_genre_vector = user_stats[["user_id"]].copy()
    user_genre_vector["user_genre_vector"] = "{}"
else:
    genre_counts = pd.crosstab(genre_exploded["user_id"], genre_exploded["genre_list"])
    genre_dist = genre_counts.div(genre_counts.sum(axis=1), axis=0).fillna(0.0)
    user_genre_vector = genre_dist.apply(
        lambda row: json.dumps({k: round(float(v), 4) for k, v in row[row > 0].to_dict().items()}),
        axis=1,
    ).reset_index(name="user_genre_vector")

# --- Language distribution per user (replaces anime Type distribution) ---
lang_frame = merged[["user_id", "original_language"]].copy()
lang_frame["original_language"] = lang_frame["original_language"].fillna("unknown").str.strip().str.lower()
lang_counts = pd.crosstab(lang_frame["user_id"], lang_frame["original_language"])
lang_dist = lang_counts.div(lang_counts.sum(axis=1), axis=0).fillna(0.0)
user_lang_distribution = lang_dist.apply(
    lambda row: json.dumps({k: round(float(v), 4) for k, v in row[row > 0].to_dict().items()}),
    axis=1,
).reset_index(name="user_language_distribution")

# --- Assemble final user features ---
user_features = user_stats.merge(user_popularity, on="user_id", how="left")
user_features = user_features.merge(user_genre_vector, on="user_id", how="left")
user_features = user_features.merge(user_lang_distribution, on="user_id", how="left")

user_features["user_avg_item_popularity"] = user_features["user_avg_item_popularity"].fillna(0.0).astype("float32")
user_features["user_genre_vector"] = user_features["user_genre_vector"].fillna("{}")
user_features["user_language_distribution"] = user_features["user_language_distribution"].fillna("{}")
user_features["user_avg_rating"] = user_features["user_avg_rating"].astype("float32")
user_features["user_rating_count"] = user_features["user_rating_count"].astype("int64")
user_features["user_rating_std"] = user_features["user_rating_std"].astype("float32")

user_features = user_features[[
    "user_id",
    "user_avg_rating",
    "user_rating_count",
    "user_rating_std",
    "user_genre_vector",
    "user_language_distribution",
    "user_avg_item_popularity",
]].sort_values("user_id").reset_index(drop=True)

USER_FEATURES_FILE.parent.mkdir(parents=True, exist_ok=True)
user_features.to_parquet(USER_FEATURES_FILE, index=False)

print(f"User feature shape: {user_features.shape}")
print(f"Saved: {USER_FEATURES_FILE}")
print(user_features.head())

User feature shape: (50000, 7)
Saved: ../../data/processed/user_features.parquet
   user_id  user_avg_rating  user_rating_count  user_rating_std  \
0        2         3.318182                 22         1.086119   
1        8         3.000000                112         1.065427   
2       10         4.230769                 13         0.725011   
3       14         3.400000                  5         1.294218   
4       27         3.647436                 78         1.206339   

                                   user_genre_vector  \
0  {"Action": 0.1406, "Adventure": 0.1094, "Comed...   
1  {"Action": 0.1258, "Adventure": 0.0806, "Anima...   
2  {"Action": 0.1538, "Adventure": 0.0769, "Anima...   
3  {"Action": 0.1818, "Crime": 0.1818, "Drama": 0...   
4  {"Action": 0.0708, "Adventure": 0.0896, "Anima...   

                          user_language_distribution  user_avg_item_popularity  
0         {"de": 0.0455, "en": 0.9091, "it": 0.0455}               1282.090942  
1  {"de": 0.0089,

In [11]:
print("=== movie_user_data.ipynb complete ===")
print(f"  interactions_clean: {OUTPUT_FILE}")
print(f"  item_features:      {ITEM_FEATURES_DIR / 'item_features.parquet'}")
print(f"  user_features:      {USER_FEATURES_FILE}")

=== movie_user_data.ipynb complete ===
  interactions_clean: ../../data/processed/interactions_clean.parquet
  item_features:      ../../data/processed/item_features.parquet
  user_features:      ../../data/processed/user_features.parquet
